<div style="background: linear-gradient(135deg, #457b9d 0%, #1d3557 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">🛠️ 02 — Construindo a U-Net do Zero</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 08 · Bloco 03 · U-Net</p>
</div>

## 🎯 Objetivos

- Escrever uma U-Net completa do zero no PyTorch
- Validar os Tensors via Forward Pass

---

## 1️⃣ Double Conv (O Bloco Básico)

A U-Net aplica duas Convoluções seguidas (Conv -> BatchNorm -> ReLU) em cada andar do 'U'.

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

print('Bloco Base criado!')

## 2️⃣ Montando a U-Net (O Lego)

Atenção extrema ao `torch.cat([x_up, skip], dim=1)`.

In [ ]:
class UNet_Mini(nn.Module):
    def __init__(self, in_c=3, out_c=1):
        super().__init__()
        self.enc1 = DoubleConv(in_c, 64)
        self.pool = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(64, 128)
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64) # 64(up) + 64(skip) = 128 in_c
        
        self.final = nn.Conv2d(64, out_c, kernel_size=1)
        
    def forward(self, x):
        # Descida
        s1 = self.enc1(x)         # Skip connection guardada
        b = self.enc2(self.pool(s1)) # Gargalo
        
        # Subida
        x_up = self.up1(b)
        x_concat = torch.cat([x_up, s1], dim=1) # CONCATENAÇÃO NO EIXO DE CANAIS
        x_out = self.dec1(x_concat)
        
        return self.final(x_out)

modelo = UNet_Mini()
img = torch.randn(1, 3, 256, 256)
print(f'Saída da U-Net: {modelo(img).shape}')

## 🏋️ Questões

### Q1. Adicione mais um andar à `UNet_Mini` acima, fazendo ela atingir 256 canais no gargalo.

### Q2. A camada final `self.final` usa kernel 1x1. Para que serve uma convolução 1x1?